# 10 Statistical Tests

## 10-1 Configuration

In [2]:
!rm -rf /content/unet-ensemble

In [3]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

Cloning into 'unet-ensemble'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (240/240), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 553 (delta 160), reused 125 (delta 63), pack-reused 313 (from 1)
Receiving objects: 100% (553/553), 188.33 KiB | 477.00 KiB/s, done.
Resolving deltas: 100% (341/341), done.ltas:  73% (249/341)


In [4]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/unet-ensemble')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import warnings
import pandas as pd
from src.eval.tests import Test

warnings.filterwarnings('ignore')

In [6]:
test = Test()

CSV_ROOT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Files/Evaluation'
OUTPUT_SAVE_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Files/Evaluation/results'

## 10-3 Comparative Assumptions Against Baseline Models

### Assumption 1: Baseline CNN detectors underperform the proposed model

#### A. Data Preparation

In [7]:
OUTPUT_FILE_NAME = 'assumption-1_proposed-vs-baselines.csv'
DATA_PATH = f'{CSV_ROOT_DIR}/02-baseline/02-baseline-analysis - #1.csv'

df = pd.read_csv(DATA_PATH)

BASELINES = {
    'Diff-ID': {
        'df'            : df,
        'iou'           : 'diff_id_iou',
        'dice'          : 'diff_id_dice',
        'kruskal_groups': {
            'iou' : ['proposed_iou'],
            'dice': ['proposed_dice'],
        },
    },
    'Two-Stream CNN': {
        'df'            : df,
        'iou'           : 'two_stream_iou',
        'dice'          : 'two_stream_dice',
        'kruskal_groups': {
            'iou' : ['proposed_iou'],
            'dice': ['proposed_dice'],
        },
    },
    'XceptionNet': {
        'df'            : df,
        'iou'           : 'xceptionnet_iou',
        'dice'          : 'xceptionnet_dice',
        'kruskal_groups': {
            'iou' : ['proposed_iou'],
            'dice': ['proposed_dice'],
        },
    },
    'SegFormer': {
        'df'            : df,
        'iou'           : 'segformer_iou',
        'dice'          : 'segformer_dice',
        'kruskal_groups': {
            'iou' : ['proposed_iou'],
            'dice': ['proposed_dice'],
        },
    },
}

#### B. Perform Testing

In [8]:
summary_10_1 = test.run_tests(
    groups    = BASELINES,
    metrics   = [('IoU', 'iou'), ('Dice', 'dice')],
    tests     = ['shapiro', 'kruskal', 'ci'],
    group_col = 'Model',
)
summary_10_1

,Metric,Model,Mean Diff,SW p,Normal,Kruskal p,Significant Kruskal p,CI Lower,CI Upper,CI Level,Favors Proposed
0,IoU,Diff-ID,0.940463,2.4070e-51,False,3.5686e-04,True,0.927627,0.952377,95.0%,True
1,Dice,Diff-ID,0.949129,3.0001e-52,False,3.5710e-04,True,0.936138,0.961000,95.0%,True
2,IoU,Two-Stream CNN,0.929617,1.8560e-50,False,2.0536e-02,True,0.915369,0.942687,95.0%,True
3,Dice,Two-Stream CNN,0.939854,1.7909e-51,False,2.0533e-02,True,0.925820,0.952715,95.0%,True
4,IoU,XceptionNet,0.958287,1.2548e-40,False,1.3856e-05,True,0.954269,0.961952,95.0%,True
5,Dice,XceptionNet,0.977439,9.9879e-48,False,1.3856e-05,True,0.974487,0.979839,95.0%,True
6,IoU,SegFormer,0.936131,8.3243e-51,False,7.1398e-03,True,0.922535,0.948329,95.0%,True
7,Dice,SegFormer,0.946193,7.1159e-52,False,7.1417e-03,True,0.933023,0.958391,95.0%,True


#### C. Save Results

In [9]:
summary_10_1.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


### Assumption 2: Forensic-conditioned pixel-wise output is spatially richer than RGB-only pixel-wise output

#### A. Data Preparation

In [8]:
OUTPUT_FILE_NAME = 'assumption-2_pixel-wise-vs-rgb-input.csv'

DIFF_ID_PATH     = f'{CSV_ROOT_DIR}/02-baseline/02-baseline-analysis - #2 - Diff-ID.csv'
TWO_STREAM_PATH  = f'{CSV_ROOT_DIR}/02-baseline/02-baseline-analysis - #2 - Two-Stream CNN.csv'
XCEPTIONNET_PATH = f'{CSV_ROOT_DIR}/02-baseline/02-baseline-analysis - #2 - XceptionNet.csv'
SEGFORMER_PATH   = f'{CSV_ROOT_DIR}/02-baseline/02-baseline-analysis - #2 - SegFormer.csv' 

diff_id_df      = pd.read_csv(DIFF_ID_PATH)
two_stream_df   = pd.read_csv(TWO_STREAM_PATH)
xceptionnet_df  = pd.read_csv(XCEPTIONNET_PATH)
segformer_df    = pd.read_csv(SEGFORMER_PATH)

BASELINES = {
    'Diff-ID': {
        'df'      : diff_id_df,
        'iou'     : 'diff_iou',
        'dice'    : 'diff_dice',
        'bf_score': 'diff_bf_score',
    },
    'Two-Stream CNN': {
        'df'      : two_stream_df,
        'iou'     : 'diff_iou',
        'dice'    : 'diff_dice',
        'bf_score': 'diff_bf_score',
    },
    'XceptionNet': {
        'df'      : xceptionnet_df,
        'iou'     : 'diff_iou',
        'dice'    : 'diff_dice',
        'bf_score': 'diff_bf_score',
    },
    'SegFormer': {
        'df'      : segformer_df,
        'iou'     : 'diff_iou',
        'dice'    : 'diff_dice',
        'bf_score': 'diff_bf_score',
    },
}

#### B. Perform Testing

In [9]:
summary_10_2 = test.run_tests(
    groups    = BASELINES,
    metrics   = [
        ('IoU',      'iou'),
        ('Dice',     'dice'),
        ('BF Score', 'bf_score'),
    ],
    tests     = ['shapiro', 'wilcoxon', 'permutation'],
    group_col = 'Baseline',
)
summary_10_2

,Metric,Baseline,Mean Diff,SW p,Normal,W p,Significant W p,Perm p,Significant Perm
0,IoU,Diff-ID,0.029185,1.3773e-51,False,1.000000e+00,False,1.0000e-04,True
1,Dice,Diff-ID,0.034037,2.7807e-52,False,1.000000e+00,False,1.0000e-04,True
2,BF Score,Diff-ID,0.026565,1.0954e-50,False,9.800420e-01,False,1.0000e-04,True
3,IoU,Two-Stream CNN,0.040032,8.1418e-51,False,1.000000e+00,False,1.0000e-04,True
4,Dice,Two-Stream CNN,0.043312,1.4943e-51,False,1.000000e+00,False,1.0000e-04,True
5,BF Score,Two-Stream CNN,0.038167,5.1386e-50,False,2.807902e-01,False,1.0000e-04,True
6,IoU,XceptionNet,0.011362,1.0201e-42,False,2.304385e-41,True,1.0000e-04,True
7,Dice,XceptionNet,0.005727,4.2081e-50,False,2.827759e-41,True,1.0000e-04,True
8,BF Score,XceptionNet,0.026074,1.2408e-43,False,1.408136e-20,True,1.0000e-04,True
9,IoU,SegFormer,0.033518,5.6096e-51,False,1.000000e+00,False,1.0000e-04,True


#### C. Save Results

In [10]:
summary_10_2.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


## 10-4 Dual-Feature vs. All-Three-Feature Assumptions

### Assumption 3: Dual-feature fusion is informationally inferior to all-three

#### A. Data Preparation

In [15]:
OUTPUT_FILE_NAME = 'assumption-3_dual-feature-vs-all.csv'

UNETPP_PATH = f'{CSV_ROOT_DIR}/03-features/03-feature-analysis - #1 - U-Net++.csv'
ENSEMBLE_PATH = f'{CSV_ROOT_DIR}/03-features//03-feature-analysis - #1 - U-Net Ensemble.csv'
ATTENUNET_PATH = f'{CSV_ROOT_DIR}/03-features//03-feature-analysis - #1 - Attention U-Net.csv'

unetpp_df = pd.read_csv(UNETPP_PATH)
ensemble_df = pd.read_csv(ENSEMBLE_PATH)
attenunet_df = pd.read_csv(ATTENUNET_PATH)

MODELS = {
    'Unet++': {
        'df': unetpp_df,
        'prnu_illum': 'diff_prnu_illum_iou',
        'prnu_freq': 'diff_prnu_freq_iou',
        'freq_illum': 'diff_freq_illum_iou',
    },
    'U-Net Ensemble':{
        'df': ensemble_df,
        'prnu_illum': 'diff_prnu_illum_iou',
        'prnu_freq': 'diff_prnu_freq_iou',
        'freq_illum': 'diff_freq_illum_iou',
    },
    'Attention U-Net': {
        'df': attenunet_df,
        'prnu_illum': 'diff_prnu_illum_iou',
        'prnu_freq': 'diff_prnu_freq_iou',
        'freq_illum': 'diff_freq_illum_iou',
    },
}

#### B. Perform Testing

In [16]:
summary_10_3 = test.run_tests(
    groups    = MODELS,
    metrics   = [
        ('PRNU + Illumination IoU',      'prnu_illum'),
        ('PRNU + Frequency IoU',         'prnu_freq'),
        ('Frequency + Illumination IoU', 'freq_illum'),
    ],
    tests     = ['shapiro', 'wilcoxon', 'permutation'],
    group_col = 'Model',
)
summary_10_3

,Metric,Model,Mean Diff,SW p,Normal,W p,Significant W p,Perm p,Significant Perm
0,PRNU + Illumination IoU,Unet++,0.002990,2.3908e-54,False,5.215027e-01,False,1.2550e-01,False
1,PRNU + Frequency IoU,Unet++,0.001685,5.0933e-54,False,8.223128e-02,False,1.6090e-01,False
2,Frequency + Illumination IoU,Unet++,0.008523,9.4755e-49,False,3.858226e-31,True,1.0000e-04,True
3,PRNU + Illumination IoU,U-Net Ensemble,-0.000834,1.0773e-37,False,9.787954e-01,False,9.8190e-01,False
4,PRNU + Frequency IoU,U-Net Ensemble,0.000714,1.3876e-53,False,7.725988e-01,False,3.8930e-01,False
5,Frequency + Illumination IoU,U-Net Ensemble,0.008117,1.4924e-43,False,4.064608e-30,True,1.0000e-04,True
6,PRNU + Illumination IoU,Attention U-Net,-0.001245,5.0247e-33,False,9.940460e-01,False,9.9640e-01,False
7,PRNU + Frequency IoU,Attention U-Net,-0.000312,2.4398e-52,False,9.980653e-01,False,5.3640e-01,False
8,Frequency + Illumination IoU,Attention U-Net,0.008221,1.2750e-46,False,1.235866e-26,True,1.0000e-04,True


#### C. Save Results

In [18]:
summary_10_3.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


### Assumption 4. Each forensic feature contributes a unique non-substitutable signal

#### A. Data Preparation

In [21]:
OUTPUT_FILE_NAME = 'assumption-4_forensic-feature-combinations.csv'

UNETPP_PATH = f'{CSV_ROOT_DIR}/03-features/03-feature-analysis - #2 - U-Net++.csv'
ATTENUNET_PATH  = f'{CSV_ROOT_DIR}/03-features/03-feature-analysis - #2 - Attention U-Net.csv'

unetpp_df = pd.read_csv(UNETPP_PATH)
attenunet_df = pd.read_csv(ATTENUNET_PATH)

MODELS = {
    'U-Net++ (PRNU + Illumination)': {
        'df'            : unetpp_df,
        'iou'           : 'unetpp_prnu_illum_iou',
        'dice'          : 'unetpp_prnu_illum_dice',
        'bf_score'      : 'unetpp_prnu_illum_bf_score',
        'kruskal_groups': {
            'iou'     : ['unetpp_prnu_freq_iou',  'unetpp_freq_illum_iou',  'unetpp_all_iou'],
            'dice'    : ['unetpp_prnu_freq_dice', 'unetpp_freq_illum_dice', 'unetpp_all_dice'],
            'bf_score': ['unetpp_prnu_freq_bf_score', 'unetpp_freq_illum_bf_score', 'unetpp_all_bf_score'],
        },
    },
    'U-Net++ (PRNU + Frequency)': {
        'df'            : unetpp_df,
        'iou'           : 'unetpp_prnu_freq_iou',
        'dice'          : 'unetpp_prnu_freq_dice',
        'bf_score'      : 'unetpp_prnu_freq_bf_score',
        'kruskal_groups': {
            'iou'     : ['unetpp_prnu_illum_iou',  'unetpp_freq_illum_iou',  'unetpp_all_iou'],
            'dice'    : ['unetpp_prnu_illum_dice', 'unetpp_freq_illum_dice', 'unetpp_all_dice'],
            'bf_score': ['unetpp_prnu_illum_bf_score', 'unetpp_freq_illum_bf_score', 'unetpp_all_bf_score'],
        },
    },
    'U-Net++ (Frequency + Illumination)': {
        'df'            : unetpp_df,
        'iou'           : 'unetpp_freq_illum_iou',
        'dice'          : 'unetpp_freq_illum_dice',
        'bf_score'      : 'unetpp_freq_illum_bf_score',
        'kruskal_groups': {
            'iou'     : ['unetpp_prnu_illum_iou',  'unetpp_prnu_freq_iou',  'unetpp_all_iou'],
            'dice'    : ['unetpp_prnu_illum_dice', 'unetpp_prnu_freq_dice', 'unetpp_all_dice'],
            'bf_score': ['unetpp_prnu_illum_bf_score', 'unetpp_prnu_freq_bf_score', 'unetpp_all_bf_score'],
        },
    },
    'U-Net++ (All)': {
        'df'            : unetpp_df,
        'iou'           : 'unetpp_all_iou',
        'dice'          : 'unetpp_all_dice',
        'bf_score'      : 'unetpp_all_bf_score',
        'kruskal_groups': {
            'iou'     : ['unetpp_prnu_illum_iou',  'unetpp_prnu_freq_iou',  'unetpp_freq_illum_iou'],
            'dice'    : ['unetpp_prnu_illum_dice', 'unetpp_prnu_freq_dice', 'unetpp_freq_illum_dice'],
            'bf_score': ['unetpp_prnu_illum_bf_score', 'unetpp_prnu_freq_bf_score', 'unetpp_freq_illum_bf_score'],
        },
    },
    'Attention U-Net (PRNU + Illumination)': {
        'df'            : attenunet_df,
        'iou'           : 'attenunet_prnu_illum_iou',
        'dice'          : 'attenunet_prnu_illum_dice',
        'bf_score'      : 'attenunet_prnu_illum_bf_score',
        'kruskal_groups': {
            'iou'     : ['attenunet_prnu_freq_iou',  'attenunet_freq_illum_iou',  'attenunet_all_iou'],
            'dice'    : ['attenunet_prnu_freq_dice', 'attenunet_freq_illum_dice', 'attenunet_all_dice'],
            'bf_score': ['attenunet_prnu_freq_bf_score', 'attenunet_freq_illum_bf_score', 'attenunet_all_bf_score'],
        },
    },
    'Attention U-Net (PRNU + Frequency)': {
        'df'            : attenunet_df,
        'iou'           : 'attenunet_prnu_freq_iou',
        'dice'          : 'attenunet_prnu_freq_dice',
        'bf_score'      : 'attenunet_prnu_freq_bf_score',
        'kruskal_groups': {
            'iou'     : ['attenunet_prnu_illum_iou',  'attenunet_freq_illum_iou',  'attenunet_all_iou'],
            'dice'    : ['attenunet_prnu_illum_dice', 'attenunet_freq_illum_dice', 'attenunet_all_dice'],
            'bf_score': ['attenunet_prnu_illum_bf_score', 'attenunet_freq_illum_bf_score', 'attenunet_all_bf_score'],
        },
    },
    'Attention U-Net (Frequency + Illumination)': {
        'df'            : attenunet_df,
        'iou'           : 'attenunet_freq_illum_iou',
        'dice'          : 'attenunet_freq_illum_dice',
        'bf_score'      : 'attenunet_freq_illum_bf_score',
        'kruskal_groups': {
            'iou'     : ['attenunet_prnu_illum_iou',  'attenunet_prnu_freq_iou',  'attenunet_all_iou'],
            'dice'    : ['attenunet_prnu_illum_dice', 'attenunet_prnu_freq_dice', 'attenunet_all_dice'],
            'bf_score': ['attenunet_prnu_illum_bf_score', 'attenunet_prnu_freq_bf_score', 'attenunet_all_bf_score'],
        },
    },
    'Attention U-Net (All)': {
        'df'            : attenunet_df,
        'iou'           : 'attenunet_all_iou',
        'dice'          : 'attenunet_all_dice',
        'bf_score'      : 'attenunet_all_bf_score',
        'kruskal_groups': {
            'iou'     : ['attenunet_prnu_illum_iou',  'attenunet_prnu_freq_iou',  'attenunet_freq_illum_iou'],
            'dice'    : ['attenunet_prnu_illum_dice', 'attenunet_prnu_freq_dice', 'attenunet_freq_illum_dice'],
            'bf_score': ['attenunet_prnu_illum_bf_score', 'attenunet_prnu_freq_bf_score', 'attenunet_freq_illum_bf_score'],
        },
    },
}

#### B. Perform Testing

In [22]:
summary_10_4 = test.run_tests(
    groups=     MODELS,
    metrics= [
        ('IoU',      'iou'),
        ('Dice',     'dice'),
        ('BF Score', 'bf_score'),
    ],
    tests = ['shapiro', 'kruskal'],
    group_col = 'Model',
)
summary_10_4

,Metric,Model,Mean Diff,SW p,Normal,Kruskal p,Significant Kruskal p
0,IoU,U-Net++ (PRNU + Illumination),0.965028,1.4830e-50,False,4.1337e-04,True
1,Dice,U-Net++ (PRNU + Illumination),0.978835,1.2391e-53,False,4.1340e-04,True
2,BF Score,U-Net++ (PRNU + Illumination),0.973882,1.3572e-51,False,2.2178e-06,True
3,IoU,U-Net++ (PRNU + Frequency),0.966332,1.7691e-49,False,4.1337e-04,True
4,Dice,U-Net++ (PRNU + Frequency),0.980498,2.3499e-53,False,4.1340e-04,True
5,BF Score,U-Net++ (PRNU + Frequency),0.972838,1.4749e-50,False,2.2178e-06,True
6,IoU,U-Net++ (Frequency + Illumination),0.959495,4.9978e-44,False,4.1337e-04,True
7,Dice,U-Net++ (Frequency + Illumination),0.977716,1.1063e-50,False,4.1340e-04,True
8,BF Score,U-Net++ (Frequency + Illumination),0.958527,7.8030e-47,False,2.2178e-06,True
9,IoU,U-Net++ (All),0.968017,4.6065e-49,False,4.1337e-04,True


#### C. Save Results

In [23]:
summary_10_4.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


### Assumption 5: U-Net++ and Attention U-Net generalize across feature configurations

#### A. Data Preparation

In [26]:
OUTPUT_FILE_NAME = 'assumption-5_model_generalization.csv'

DATA_PATH = f'{CSV_ROOT_DIR}/03-features/03-feature-analysis - #3.csv'
df = pd.read_csv(DATA_PATH)

MODELS = {
    'U-Net++': {
        'df'            : df,
        'prnu_illum'    : 'unetpp_prnu_illum_iou',
        'prnu_freq'     : 'unetpp_prnu_freq_iou',
        'freq_illum'    : 'unetpp_freq_illum_iou',
        'all'           : 'unetpp_all_iou',
        'kruskal_groups': {
            'prnu_illum': ['attenunet_prnu_illum_iou'],
            'prnu_freq' : ['attenunet_prnu_freq_iou'],
            'freq_illum': ['attenunet_freq_illum_iou'],
            'all'       : ['attenunet_all_iou'],
        },
    },
    'Attention U-Net': {
        'df'            : df,
        'prnu_illum'    : 'attenunet_prnu_illum_iou',
        'prnu_freq'     : 'attenunet_prnu_freq_iou',
        'freq_illum'    : 'attenunet_freq_illum_iou',
        'all'           : 'attenunet_all_iou',
        'kruskal_groups': {
            'prnu_illum': ['unetpp_prnu_illum_iou'],
            'prnu_freq' : ['unetpp_prnu_freq_iou'],
            'freq_illum': ['unetpp_freq_illum_iou'],
            'all'       : ['unetpp_all_iou'],
        },
    },
}

#### B. Perform Testing

In [27]:
summary_10_5 = test.run_tests(
    groups = MODELS,
    metrics = [
        ('PRNU + Illumination', 'prnu_illum'),
        ('PRNU + Frequency', 'prnu_freq'),
        ('Frequency + Illumination', 'freq_illum'),
        ('All Features', 'all'),
    ],
    tests = ['shapiro', 'kruskal'],
    group_col = 'Model'
)
summary_10_5

,Metric,Model,Mean Diff,SW p,Normal,Kruskal p,Significant Kruskal p
0,PRNU + Illumination,U-Net++,0.965028,1.4830e-50,False,8.1093e-01,False
1,PRNU + Frequency,U-Net++,0.966332,1.7691e-49,False,9.6530e-01,False
2,Frequency + Illumination,U-Net++,0.959495,4.9978e-44,False,6.5705e-01,False
3,All Features,U-Net++,0.968017,4.6065e-49,False,3.9600e-01,False
4,PRNU + Illumination,Attention U-Net,0.967752,6.8868e-49,False,8.1093e-01,False
5,PRNU + Frequency,Attention U-Net,0.966820,2.1392e-49,False,9.6530e-01,False
6,Frequency + Illumination,Attention U-Net,0.958287,1.2548e-40,False,6.5705e-01,False
7,All Features,Attention U-Net,0.966507,1.9454e-48,False,3.9600e-01,False


#### C. Save Results

In [28]:
summary_10_5.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


### Assumption 6: Full three-feature configuration achieves superior localization

#### A. Data Preparation

In [31]:
OUTPUT_FILE_NAME = 'assumption-6_three-feature-superiority.csv'

UNETPP_PATH = f'{CSV_ROOT_DIR}/03-features/03-feature-analysis - #4 - U-Net++.csv'
ENSEMBLE_PATH = f'{CSV_ROOT_DIR}/03-features//03-feature-analysis - #4 - U-Net Ensemble.csv'
ATTENUNET_PATH = f'{CSV_ROOT_DIR}/03-features//03-feature-analysis - #4 - Attention U-Net.csv'

unetpp_df = pd.read_csv(UNETPP_PATH)
ensemble_df = pd.read_csv(ENSEMBLE_PATH)
attenunet_df = pd.read_csv(ATTENUNET_PATH)

MODELS = {
    'Unet++': {
        'df': unetpp_df,
        'prnu_illum': 'diff_prnu_illum_dice',
        'prnu_freq': 'diff_prnu_freq_dice',
        'freq_illum': 'diff_freq_illum_dice',
    },
    'U-Net Ensemble':{
        'df': ensemble_df,
        'prnu_illum': 'diff_prnu_illum_dice',
        'prnu_freq': 'diff_prnu_freq_dice',
        'freq_illum': 'diff_freq_illum_dice',
    },
    'Attention U-Net': {
        'df': attenunet_df,
        'prnu_illum': 'diff_prnu_illum_dice',
        'prnu_freq': 'diff_prnu_freq_dice',
        'freq_illum': 'diff_freq_illum_dice',
    },
}

#### B. Perform Testing

In [32]:
summary_10_6 = test.run_tests(
    groups    = MODELS,
    metrics   = [
        ('PRNU + Illumination Dice',      'prnu_illum'),
        ('PRNU + Frequency Dice',         'prnu_freq'),
        ('Frequency + Illumination Dice', 'freq_illum'),
    ],
    tests     = ['shapiro', 'wilcoxon', 'permutation'],
    group_col = 'Model',
)
summary_10_6

,Metric,Model,Mean Diff,SW p,Normal,W p,Significant W p,Perm p,Significant Perm
0,PRNU + Illumination Dice,Unet++,0.002991,3.4835e-55,False,5.258497e-01,False,1.2910e-01,False
1,PRNU + Frequency Dice,Unet++,0.001328,3.0929e-55,False,8.570465e-02,False,1.8410e-01,False
2,Frequency + Illumination Dice,Unet++,0.004110,6.5290e-53,False,5.573936e-31,True,1.0000e-04,True
3,PRNU + Illumination Dice,U-Net Ensemble,-0.000502,5.4317e-43,False,9.790642e-01,False,9.8430e-01,False
4,PRNU + Frequency Dice,U-Net Ensemble,0.000799,2.5186e-55,False,7.730464e-01,False,3.9160e-01,False
5,Frequency + Illumination Dice,U-Net Ensemble,0.003932,8.6587e-51,False,4.672764e-30,True,1.0000e-04,True
6,PRNU + Illumination Dice,Attention U-Net,-0.000673,1.9270e-33,False,9.938092e-01,False,9.9650e-01,False
7,PRNU + Frequency Dice,Attention U-Net,0.000161,9.4997e-55,False,9.979878e-01,False,4.9750e-01,False
8,Frequency + Illumination Dice,Attention U-Net,0.003652,1.3671e-52,False,1.562844e-26,True,1.0000e-04,True


#### C. Save Results

In [33]:
summary_10_6.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


## 10-5 Ensemble Mask Fusion Assumptions

### Assumption 7: Combining UNetPP and AttUNet masks produces more reliable predictions than either model alone

#### A. Data Preparation

In [40]:
OUTPUT_FILE_NAME = 'assumption-7_single-model-vs-ensemble.csv'

DATA_PATH = f'{CSV_ROOT_DIR}/04-ensemble/04-ensemble-analysis - #1-#2.csv'
df = pd.read_csv(DATA_PATH)

df['diff_ensemble_unetpp_iou']      = df['ensemble_iou']      - df['unetpp_iou']
df['diff_ensemble_unetpp_dice']     = df['ensemble_dice']     - df['unetpp_dice']
df['diff_ensemble_unetpp_bf_score'] = df['ensemble_bf_score'] - df['unetpp_bf_score']

df['diff_ensemble_attenunet_iou']      = df['ensemble_iou']      - df['attenunet_iou']
df['diff_ensemble_attenunet_dice']     = df['ensemble_dice']     - df['attenunet_dice']
df['diff_ensemble_attenunet_bf_score'] = df['ensemble_bf_score'] - df['attenunet_bf_score']

MODELS = {
    'U-Net++': {
        'df'      : df,
        'iou'     : 'diff_ensemble_unetpp_iou',
        'dice'    : 'diff_ensemble_unetpp_dice',
        'bf_score': 'diff_ensemble_unetpp_bf_score',
    },
    'Attention U-Net': {
        'df'      : df,
        'iou'     : 'diff_ensemble_attenunet_iou',
        'dice'    : 'diff_ensemble_attenunet_dice',
        'bf_score': 'diff_ensemble_attenunet_bf_score',
    },
}

#### B. Perform Testing

In [42]:
summary_10_7 = test.run_tests(
    groups      = MODELS,
    metrics     = [
        ('IoU', 'iou'),
        ('Dice', 'dice'),
        ('BF Score', 'bf_score'),
    ],
    tests       = ['shapiro', 'wilcoxon', 'permutation'],
    group_col   = 'Model'
)
summary_10_7

,Metric,Model,Mean Diff,SW p,Normal,W p,Significant W p,Perm p,Significant Perm
0,IoU,U-Net++,0.001631,4.5807e-55,False,2.502528e-04,True,5.3000e-03,True
1,Dice,U-Net++,0.001340,5.7894e-56,False,2.374690e-04,True,9.0000e-03,True
2,BF Score,U-Net++,0.002818,4.5358e-52,False,1.614423e-03,True,5.0000e-03,True
3,IoU,Attention U-Net,0.003141,6.1661e-52,False,1.823113e-28,True,1.0000e-04,True
4,Dice,Attention U-Net,0.002075,2.0359e-55,False,3.028296e-28,True,1.0000e-04,True
5,BF Score,Attention U-Net,0.004303,1.3723e-45,False,2.353001e-09,True,1.0000e-04,True


#### C. Save Results

In [43]:
summary_10_7.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.


### Assumption 8: Ensemble fusion reduces false positives and false negatives, improving localization precision

#### A. Data Preparation

In [44]:
OUTPUT_FILE_NAME = 'assumption-8_esemble-reduce-fp-fn.csv'

DATA_PATH = f'{CSV_ROOT_DIR}/04-ensemble/04-ensemble-analysis - #1-#2.csv'
df = pd.read_csv(DATA_PATH)

df['diff_ensemble_unetpp_iou']      = df['ensemble_iou']      - df['unetpp_iou']
df['diff_ensemble_unetpp_dice']     = df['ensemble_dice']     - df['unetpp_dice']
df['diff_ensemble_unetpp_bf_score'] = df['ensemble_bf_score'] - df['unetpp_bf_score']

df['diff_ensemble_attenunet_iou']      = df['ensemble_iou']      - df['attenunet_iou']
df['diff_ensemble_attenunet_dice']     = df['ensemble_dice']     - df['attenunet_dice']
df['diff_ensemble_attenunet_bf_score'] = df['ensemble_bf_score'] - df['attenunet_bf_score']

MODELS = {
    'U-Net++': {
        'df'      : df,
        'iou'     : 'diff_ensemble_unetpp_iou',
        'dice'    : 'diff_ensemble_unetpp_dice',
        'bf_score': 'diff_ensemble_unetpp_bf_score',
    },
    'Attention U-Net': {
        'df'      : df,
        'iou'     : 'diff_ensemble_attenunet_iou',
        'dice'    : 'diff_ensemble_attenunet_dice',
        'bf_score': 'diff_ensemble_attenunet_bf_score',
    },
}

#### B. Perform Testing

In [45]:
summary_10_8 = test.run_tests(
    groups      = MODELS,
    metrics     = [
        ('IoU', 'iou'),
        ('Dice', 'dice'),
        ('BF Score', 'bf_score'),
    ],
    tests       = ['shapiro', 'wilcoxon', 'permutation'],
    group_col   = 'Model'
)
summary_10_8

,Metric,Model,Mean Diff,SW p,Normal,W p,Significant W p,Perm p,Significant Perm
0,IoU,U-Net++,0.001631,4.5807e-55,False,2.502528e-04,True,5.7000e-03,True
1,Dice,U-Net++,0.001340,5.7894e-56,False,2.374690e-04,True,1.1100e-02,True
2,BF Score,U-Net++,0.002818,4.5358e-52,False,1.614423e-03,True,3.2000e-03,True
3,IoU,Attention U-Net,0.003141,6.1661e-52,False,1.823113e-28,True,1.0000e-04,True
4,Dice,Attention U-Net,0.002075,2.0359e-55,False,3.028296e-28,True,1.0000e-04,True
5,BF Score,Attention U-Net,0.004303,1.3723e-45,False,2.353001e-09,True,1.0000e-04,True


#### C. Save Results

In [46]:
summary_10_8.to_csv(f'{OUTPUT_SAVE_DIR}/{OUTPUT_FILE_NAME}', index=False)
print('Saved.')

Saved.
